In [124]:
import pandas as pd
from tqdm import tqdm
from glob import glob
from sklearn.neighbors import NearestNeighbors
import numpy as np

In [4]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("tsiaras/uk-road-safety-accidents-and-vehicles")

print("Path to dataset files:", path)

Download already complete (149163721 bytes).
Extracting files...
Path to dataset files: C:\Users\M\.cache\kagglehub\datasets\tsiaras\uk-road-safety-accidents-and-vehicles\versions\3


In [125]:
acc = pd.read_csv("../data/raw/Accident_Information.csv")
acc['Date'] = pd.to_datetime(acc['Date'], format='%Y-%m-%d')

C:\Users\M\AppData\Local\Temp\ipykernel_18468\1717480648.py:1: DtypeWarning: Columns (0: Accident_Index) have mixed types. Specify dtype option on import or set low_memory=False.
  acc = pd.read_csv("../data/raw/Accident_Information.csv")


In [127]:
veh = pd.read_csv("../data/raw/Vehicle_Information.csv", encoding="cp1252")

steps:
1. load the stations weather data 
2. load the stations meta data
3. for each row get 2 things: date as year and month, longitude and latitidue and 
4. run the algorithm that will get the nearest station to this accident
5. use date to get weather data

In [128]:
stations_list = []
for file in glob("../data/raw/stations_data/*"):
    name = file.split("\\")[1].split(".")[0].replace(" ","_").strip()
    print(name)
    name = pd.read_csv(file)
    name = name.iloc[:, 1:]
    name = name.drop("Date", axis=1)
    stations_list.append(name)
    

Aberporth
Armagh
Ballypatrick_Forest
Bradford
Braemar
Camborne
Cambridge_NIAB
Cardiff_Bute_Park
Chivenor
Cwmystwyth
Dunstaffnage
Durham
Eastbourne
Eskdalemuir
Heathrow
Hurn
Lerwick
Leuchars
Lowestoft
Manston
Nairn
Newton_Rigg
Oxford
Paisley
Ringway
Ross-on-Wye
Shawbury
Sheffield
Southampton
Stornoway_Airport
Sutton_Bonington
Tiree
Valley
Waddington
Whitby
Wick_Airport
Yeovilton


In [129]:
meta = pd.read_csv("../data/raw/stations_metadata/stations.csv")

In [130]:
X = np.array(meta[["lat", "lon"]])

In [131]:
nbrs = NearestNeighbors(n_neighbors=1, algorithm='ball_tree').fit(X)

In [132]:
valid_mask = acc[["Latitude", "Longitude"]].notna().all(axis=1)
valid_coords = acc.loc[valid_mask, ["Latitude", "Longitude"]]

distances, indices = nbrs.kneighbors(np.array(valid_coords[["Latitude", "Longitude"]]))


In [109]:
# 0 is the first element in the big array
#get date of the first element
acc_year = acc.iloc[0]["Date"].year
acc_month = acc.iloc[0]["Date"].month

station_df = stations_list[indices[0].item()]
station_df[(station_df['Year'] == acc_year) & (station_df['Month'] == acc_month)].reset_index(drop = True)

,Year,Month,Tmax,Tmin,AF,Rain,Sun,status,Tmean
0,2005,1,9.5,3.8,1.0,21.6,72.1,NaN,6.65


In [134]:
results = []

for i in tqdm(range(len(acc)-1)):
    row = acc.iloc[i]
    
    acc_year  = row["Date"].year
    acc_month = row["Date"].month
    
    station_df = stations_list[indices[i].item()]
    matched = station_df[(station_df["Year"] == acc_year) & (station_df["Month"] == acc_month)].reset_index(drop=True)
    
    if len(matched) == 0:
        results.append({"Accident_Index": row["Accident_Index"]})
        continue
    
    record = matched.iloc[0].to_dict()
    record["Accident_Index"] = row["Accident_Index"] 
    results.append(record)

  0%|          | 0/2047255 [00:00<?, ?it/s]

100%|█████████▉| 2047081/2047255 [21:22<00:00, 1595.60it/s]


IndexError: index 2047081 is out of bounds for axis 0 with size 2047081

In [135]:
weather_result_df = pd.DataFrame(results)
weather_result_df["Accident_Index"] = weather_result_df["Accident_Index"].astype(str)


# Merge with original acc df
#acc_merged = acc.merge(weather_result_df, on="Accident_Index", how="left")

In [136]:
weather_result_df

,Year,Month,Tmax,Tmin,AF,Rain,Sun,status,Tmean,Accident_Index
0,2005.0,1.0,9.5,3.8,1.0,21.6,72.1,NaN,6.65,200501BS00001
1,2005.0,1.0,9.5,3.8,1.0,21.6,72.1,NaN,6.65,200501BS00002
2,2005.0,1.0,9.5,3.8,1.0,21.6,72.1,NaN,6.65,200501BS00003
3,2005.0,1.0,9.5,3.8,1.0,21.6,72.1,NaN,6.65,200501BS00004
4,2005.0,1.0,9.5,3.8,1.0,21.6,72.1,NaN,6.65,200501BS00005
...,...,...,...,...,...,...,...,...,...,...
2047076,2017.0,3.0,9.6,1.9,7.0,243.8,103.7,NaN,5.75,2017982103817
2047077,2017.0,5.0,16.0,6.2,1.0,102.8,200.9,NaN,11.10,2017982104417
2047078,2017.0,5.0,16.0,6.2,1.0,102.8,200.9,NaN,11.10,2017982104617
2047079,2017.0,5.0,16.0,6.2,1.0,102.8,200.9,NaN,11.10,2017982104717


In [137]:
acc_merged = acc.merge(veh, on="Accident_Index", how="left").merge(weather_result_df, on="Accident_Index", how="left")

In [141]:
acc_merged["Accident_Index"] = acc_merged["Accident_Index"].astype(str)

In [138]:
acc_merged.head()

,Accident_Index,1st_Road_Class,1st_Road_Number,2nd_Road_Class,2nd_Road_Number,Accident_Severity,Carriageway_Hazards,Date,Day_of_Week,Did_Police_Officer_Attend_Scene_of_Accident,...,Year_y,Year,Month,Tmax,Tmin,AF,Rain,Sun,status,Tmean
0,200501BS00001,A,3218.0,NaN,0.0,Serious,NaN,2005-01-04,Tuesday,1.0,...,NaN,2005.0,1.0,9.5,3.8,1.0,21.6,72.1,NaN,6.65
1,200501BS00002,B,450.0,C,0.0,Slight,NaN,2005-01-05,Wednesday,1.0,...,2005.0,2005.0,1.0,9.5,3.8,1.0,21.6,72.1,NaN,6.65
2,200501BS00003,C,0.0,NaN,0.0,Slight,NaN,2005-01-06,Thursday,1.0,...,2005.0,2005.0,1.0,9.5,3.8,1.0,21.6,72.1,NaN,6.65
3,200501BS00004,A,3220.0,NaN,0.0,Slight,NaN,2005-01-07,Friday,1.0,...,2005.0,2005.0,1.0,9.5,3.8,1.0,21.6,72.1,NaN,6.65
4,200501BS00005,Unclassified,0.0,NaN,0.0,Slight,NaN,2005-01-10,Monday,1.0,...,2005.0,2005.0,1.0,9.5,3.8,1.0,21.6,72.1,NaN,6.65


In [139]:
weather_result_df.to_parquet("../data/interim/weather_data.parquet",  engine="fastparquet", index=False)

In [142]:
acc_merged.to_parquet("../data/interim/merged_data.parquet",  engine="fastparquet", index=False)